# Análise Exploratória de Dados - Obras Públicas de Caruaru/PE

Este notebook apresenta uma análise exploratória completa dos dados de obras públicas extraídos do Portal da Transparência da Prefeitura de Caruaru/PE.

- Autor: Tiago Henrique
- Data: Novembro 2025
- Dataset: 419 obras públicas municipais extraídos em 22/11/2025

## 1. Carregaento e Inspeção Inicial dos Dados

In [1]:
import os
import pandas as pd
import numpy as np
import json
from datetime import datetime
import re

In [2]:
print("="*80)
print("CARREGARMENTO E INSPEÇÃO")
print("="*80)

base_url = os.path.dirname(os.path.dirname(os.path.abspath(__name__)))
filename = 'public_works_test.json'
data_path = os.path.join(base_url, filename)

with open(data_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

print(f"\n Dataset carregado com sucesso!")
print(f" · Total de registros: {len(data)}")
print(f" · Estrutura: {type(data)}")

df = pd.DataFrame(data)

print(f"\n Estrutura do Dataframe:")
print(f" · Shape: {df.shape}")
print(f" · Colunas: {len(df.columns)}")
print(f" · Memória Utilizada: {df.memory_usage(deep=True).sum() / 1024**2:2f} MB")

print(f"\nColunas Disponíveis:")
for i, col in enumerate(df.columns, 1):
    null_count = df[col].isnull().sum()
    print(f"  {i:2d}. {col:<30} - Nulos: {null_count} ({null_count/len(df)*100:.1f}%)")



CARREGARMENTO E INSPEÇÃO

 Dataset carregado com sucesso!
 · Total de registros: 419
 · Estrutura: <class 'list'>

 Estrutura do Dataframe:
 · Shape: (419, 11)
 · Colunas: 11
 · Memória Utilizada: 0.810386 MB

Colunas Disponíveis:
   1. source_url                     - Nulos: 0 (0.0%)
   2. extraction_date                - Nulos: 0 (0.0%)
   3. general_info                   - Nulos: 0 (0.0%)
   4. contract                       - Nulos: 0 (0.0%)
   5. contracted                     - Nulos: 0 (0.0%)
   6. fiscal_year_expenditures       - Nulos: 0 (0.0%)
   7. contract_amendment             - Nulos: 0 (0.0%)
   8. cumulative_amount_paid         - Nulos: 0 (0.0%)
   9. all_documents                  - Nulos: 0 (0.0%)
  10. work_location                  - Nulos: 0 (0.0%)
  11. extraction_summary             - Nulos: 0 (0.0%)


In [3]:
df.head()

,source_url,extraction_date,general_info,contract,contracted,fiscal_year_expenditures,contract_amendment,cumulative_amount_paid,all_documents,work_location,extraction_summary
0,https://caruaru.pe.gov.br/obras/concorrencia-n...,2025-12-22T08:57:17.034673,{'paired_data': [{'category': 'Modalidade/nº d...,"{'paired_data': [{'category': 'Nº', 'value': '...","{'paired_data': [{'category': '19/2024', 'valu...",{'paired_data': [{'category': 'Valor Médio Acu...,"{'paired_data': [{'category': 'Prazo Aditado',...",{'paired_data': [{'category': 'Valor pago acum...,{'paired_data': [{'category': 'Projeto Básico'...,[https://caruaru.pe.gov.br/wp-content/uploads/...,"{'total_sections_processed': 8, 'successful_ex..."
1,https://caruaru.pe.gov.br/obras/dispensa-001-2...,2025-12-22T08:57:17.045117,{'paired_data': [{'category': 'Modalidade/nº d...,"{'paired_data': [{'category': 'Nº', 'value': '...","{'paired_data': [{'category': '012/2024', 'val...",{'paired_data': [{'category': 'Valor Médio Acu...,"{'paired_data': [{'category': 'Prazo Aditado',...",{'paired_data': [{'category': 'Valor pago acum...,{'paired_data': [{'category': 'Processo na ínt...,[https://caruaru.pe.gov.br/wp-content/uploads/...,"{'total_sections_processed': 8, 'successful_ex..."
2,https://caruaru.pe.gov.br/obras/pregao-eletron...,2025-12-22T08:57:17.053175,{'paired_data': [{'category': 'Modalidade/nº d...,"{'paired_data': [{'category': 'Nº', 'value': '...","{'paired_data': [{'category': '075/2024', 'val...",{'paired_data': [{'category': 'Valor Médio Acu...,"{'paired_data': [{'category': 'Prazo Aditado',...",{'paired_data': [{'category': 'Valor pago acum...,{'paired_data': [{'category': 'Processo na ínt...,[https://caruaru.pe.gov.br/wp-content/uploads/...,"{'total_sections_processed': 8, 'successful_ex..."
3,https://caruaru.pe.gov.br/obras/dispensa-eletr...,2025-12-22T08:57:17.060230,{'paired_data': [{'category': 'Modalidade/nº d...,"{'paired_data': [{'category': 'Nº', 'value': '...","{'paired_data': [{'category': '172/2025', 'val...",{'paired_data': [{'category': 'Valor Médio Acu...,"{'paired_data': [{'category': 'Prazo Aditado',...",{'paired_data': [{'category': 'Valor pago acum...,{'paired_data': [{'category': 'Termo de Referê...,[https://caruaru.pe.gov.br/wp-content/uploads/...,"{'total_sections_processed': 8, 'successful_ex..."
4,https://caruaru.pe.gov.br/obras/concorrencia-e...,2025-12-22T08:57:17.068709,{'paired_data': [{'category': 'Modalidade/nº d...,"{'paired_data': [{'category': 'Nº', 'value': '...","{'paired_data': [{'category': '089/2025', 'val...",{'paired_data': [{'category': 'Valor Médio Acu...,"{'paired_data': [{'category': 'Prazo Aditado',...",{'paired_data': [{'category': 'Valor pago acum...,{'paired_data': [{'category': 'Documentos na Í...,[https://caruaru.pe.gov.br/wp-content/uploads/...,"{'total_sections_processed': 8, 'successful_ex..."


### 2.1 Funções Auxiliares de Limpeza dos Dados

In [4]:
def extract_section_pairs(row, section_name):
    """Função genérica para extrair qualquer seção pareada"""
    if pd.isna(row) or "paired_data" not in row:
        return pd.Series({"valor": None, "data": None})

    pairs = row["paired_data"]
    result = {}

    for pair in pairs:
        category = pair.get("category", "").lower()
        value = pair.get("value", "")
        clean_category = re.sub(r'[^\w\s-]', '', category.strip().lower().replace(' ', '_'))
        # Mapeamento específico por seção
        result[f"{section_name}_{category}"] = value

    return pd.Series(result)

In [5]:
def clean_currency(value):
    """Converte valores monetários para float"""
    if pd.isna(value) or value == "" or value == "-":
        return np.nan

    clean_val = str(value).replace("R$", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(clean_val)
    except:
        return np.nan


In [6]:
def clean_percentage(value):
    """Converte percentuais para float"""
    if pd.isna(value) or value == "" or value == "-":
        return np.nan
    clena_val = str(value).replace("%", "").replace(",", ".").strip()
    try:
        return float(clena_val)
    except:
        return np.nan
        

In [7]:
def parse_date(date_str, format="%d/%m/%Y"):
    """Converte strings para data para datetime"""
    if pd.isna(date_str) or date_str == "" or date_str == "-":
        return pd.NaT
    try:
        return pd.to_datetime(date_str, format=format, errors="coerce")
    except:
        return pd.NaT
        

In [8]:
def extract_days(term_text):
    """Extrai o número de dias do texto de prazo"""
    if pd.isna(term_text) or term_text == "" or term_text == "-":
        return np.nan
        
    match = re.search(r"(\d+)\s*dias?", str(term_text), re.IGNORECASE)

    if match:
        return int(match.group(1))

    return np.nan
    

In [9]:
def clean_text(text):
    """Limpa e padroniza texto"""
    if pd.isna(text) or text == "":
        return None

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text if text and text != "-" else None
    

In [10]:
def extract_modality_type(licitacao_text):
    """Extrai o tipo de modalidade de licitação"""
    if pd.isna(licitacao_text) or licitacao_text == "":
        return None

    first_word = str(licitacao_text).split()[0].upper()

    mapping = {
        "PREGÃO": "PREGÃO",
        "PREG": "PREGÃO",
        "CP": "CARTA PROPOSTA",
        "TP": "TOMADA DE PREÇO",
        "TOMADA": "TOMADA DE PREÇO",
        "CONCORRÊNCIA": "CONCORRÊNCIA",
        "CV": "CONVITE",
        "CONVITE": "CONVITE",
        "DISPENSA": "DISPENSA",
        "DISP": "DISPENSA",
        "INEXIGIBILIDADE": "INEXIGIBILIDADE",
    }
    
    return mapping.get(first_word, first_word)


In [11]:
def extract_document_type(doc_text):
    """Identifica CPF ou CNPJ"""
    if pd.isna(doc_text) or doc_text == "":
        return None

    doc_clean = str(doc_text).replace(".", "").replace("-","").replace("/", "").strip()
    if len(doc_clean) == 11:
        return "CPF"
    elif len(doc_clean) == 14:
        return "CNPJ"
    else:
        return "IRRGULAR"
        

### 2.2 Expansão dos dados aninhados

In [12]:
# 1. Extrair informações de general_info
def extract_general_info(row): return extract_section_pairs(row, "general_info")
def extract_contract(row): return extract_section_pairs(row, "contract")
def extract_contracted(row): return extract_section_pairs(row, "contracted")
def extract_fiscal_year_expenditures(row): return extract_section_pairs(row, "fiscal_year_expenditures")
def extract_contract_amendment(row): return extract_section_pairs(row, "contract_amendment")
def extract_contract_cumulative_amount_paid(row): return extract_section_pairs(row, "cumulative_amount_paid")
def extract_work_location(row): return extract_section_pairs(row, "work_location")

In [13]:
general_info_df = df["general_info"].apply(extract_general_info)
contract_df = df["contract"].apply(extract_contract) 
contracted_df = df["contracted"].apply(extract_contracted)

contracted_df = contracted_df.rename(columns={
    "numero_contrato": "numero_contrato_referencia",
    "cnpj_contratada": "cnpj_contratada",
    "razao_social_contratada":"razao_social_contratada",
    "data_inicio_contrato": "data_icicio_contratada",
})
#general_info_df.head()
#contract_df.head()
contracted_df.sample(10)

,contracted_19/2024,contracted_23/07/2024,contracted_012/2024,contracted_09/07/2024,contracted_075/2024,contracted_27/06/2024,contracted_172/2025,contracted_08/07/2025,contracted_089/2025,contracted_2 meses,...,contracted_075/2018,contracted_09/09/2019,contracted_115/2019,contracted_110/2019,contracted_111/2019,contracted_126/2019,contracted_105/2019,contracted_31/10/2019,contracted_089/2019,contracted_16/12/2019
86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
350,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
234,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 2.2.1 Funções específicas de extração por seções

##### Contracted

In [14]:
# contracted
def extract_contracted(row):
    """
    Padrão:
    - par 0: category = no contrato, value = CNPJ
    - par 1: category = data início, value = razão social
    """

    if row is None or (isinstance(row, float) and pd.isna(row)) or "paired_data" not in row:
        return pd.Series({

            "numero_contrato_referencia": None,
            "cnpj_contratada": None,
            "razao_social_contratada": None,
            "data_icicio_contratada": None,
        })

    pairs = row["paired_data"]
    result = {
        "numero_contrato_referencia": None,
        "cnpj_contratada": None,
        "razao_social_contratada": None,
        "data_icicio_contratada": None,
    }

    if len(pairs) > 0:
        result["numero_contrato_referencia"] = pairs[0].get("category")
        result["cnpj_contratada"] = pairs[0].get("value")

    if len(pairs) > 1:
        result["data_inicio_contratada"] = pairs[1].get("category")
        result["razao_social_contratada"] = pairs[1].get("value")

    return pd.Series(result)
    

In [15]:
contracted_df = df["contracted"].apply(extract_contracted)

contracted_df.head()

,numero_contrato_referencia,cnpj_contratada,razao_social_contratada,data_icicio_contratada,data_inicio_contratada
0,19/2024,00.654.704/0001-88,A B L ENGENHARIA COMERCIO E REPRESENTACAO LTDA,None,23/07/2024
1,012/2024,07.408.234/0001-11,L. & R. SANTOS CONSTRUCOES LTDA,None,09/07/2024
2,075/2024,19.795.706/0001-15,"ENOVE ENGENHARIA, COMERCIO DE MATERIAS ELETRIC...",None,27/06/2024
3,172/2025,51.323.527/0001-98,PREVENIR EXTINTORES E INSTALAÇÕES LTDA,None,08/07/2025
4,089/2025,34.346.587/0001-07,ENGETEC SERVICOS DE ENGENHARIA LTDA,None,2 MESES


In [16]:
contract_df = df["contract"].apply(extract_contract) 
contract_df.head()

,contract_nº,contract_data ínicio,contract_prazo,contract_valor contratado (r$),contract_data conclusão / paralisação
0,19/2024,23/07/2024,12 MESES,"3.367.764,05",-
1,012/2024,09/07/2024,12 MESES,"741.995,10",-
2,075/2024,27/06/2024,12 MESES,"12.000.000,00",-
3,172/2025,08/07/2025,2 MESES,"116.980,00",-
4,089/2025,2 MESES,"226.403,54",-,NaN


##### Fiscal Year Expenditures

In [17]:
def extract_fiscal_year_expenditures(row):
    """
    Padrão:
    - Valor Médio Acumulado R$
    - Valor pago Acumulado no peírodo R$
    - Valor pago Acumulado no exercício R$
    """

    if row is None or (isinstance(row, float) and pd.isna(row)) or "paired_data" not in row:
        return pd.Series({
            "valor_medio_acumulado": None,
            "valor_pago_periodo": None,
            "valor_pago_exercicio": None,
        })

    result = {
        "valor_medio_acumulado": None,
        "valor_pago_periodo": None,
        "valor_pago_exercicio": None,
    }
    
    for pair in row["paired_data"]:
        cat = str(pair.get("category", "")).lower()
        val = pair.get("value", "")

        if "valor médio acumulado" in cat or "valor medio acumulado" in cat:
            result["valor_medio_acumulado"] = val

        elif "período" in cat or "periodo" in cat:
            result["valor_pago_periodo"] = val

        elif "exercício" in cat or "exercicio" in cat:
            result["valor_pago_exercicio"] = val

    return pd.Series(result)
    

In [18]:
df["fiscal_year_expenditures"]

0      {'paired_data': [{'category': 'Valor Médio Acu...
1      {'paired_data': [{'category': 'Valor Médio Acu...
2      {'paired_data': [{'category': 'Valor Médio Acu...
3      {'paired_data': [{'category': 'Valor Médio Acu...
4      {'paired_data': [{'category': 'Valor Médio Acu...
                             ...                        
414    {'paired_data': [{'category': 'Valor Médio Acu...
415    {'paired_data': [{'category': 'Valor Médio Acu...
416    {'paired_data': [{'category': 'Valor Médio Acu...
417    {'paired_data': [{'category': 'Valor Médio Acu...
418    {'paired_data': [{'category': 'Valor Médio Acu...
Name: fiscal_year_expenditures, Length: 419, dtype: object

In [19]:
fiscal_df = df["fiscal_year_expenditures"].apply(extract_fiscal_year_expenditures)
fiscal_df.head()

,valor_medio_acumulado,valor_pago_periodo,valor_pago_exercicio
0,"413.074,68","344.629,58","344.629,58"
1,"627.104,97","324.579,24","324.579,24"
2,"5.280.000,00","5.280.000,00","5.280.000,00"
3,0,0,0
4,0,0,0


##### Contract Amendment

In [20]:
amendment_df = df["contract_amendment"].apply(extract_contract_amendment)
amendment_df.sample(10)

,contract_amendment_prazo aditado,contract_amendment_valor aditado acumulado (r$)
174,03/07/2023,"508.862,75"
361,11/10/2019 (03 MESES),"569.085,59"
211,08/06/2023 - 277 dias,"1.458.426,16"
99,22/06/2025 (24 MESES),"464.728,38"
87,16/06/2024 (06 MESES),"487.881,82"
53,90 DIAS (15/04/2024) 03 MESES,-
160,246 dias,07/06/2024
196,22/02/2023 (03 MESES),-
416,-,-
180,02/07/2023 (37 MESES),"488.542,92"


##### Cumulative Amount Paid

In [21]:
cumulative_df = df["cumulative_amount_paid"].apply(extract_contract_cumulative_amount_paid)
cumulative_df.head()

,cumulative_amount_paid_valor pago acumulado
0,"344.629,58"
1,"324.579,24"
2,"5.280.000,00"
3,0
4,0


### 2.3 Tratamento dos dados de informações gerais

In [22]:
general_info_df.columns

Index(['general_info_modalidade/nº da licitação',
       'general_info_descrição da obra', 'general_info_situação',
       'general_info_etapa da obra', 'general_info_percentual concluído',
       'general_info_responsável pela inexecução temporária do objeto',
       'general_info_motivo',
       'general_info_data prevista para reinício da obra'],
      dtype='object')

In [23]:
general_info_df.columns = [
    "modalidade_licitacao",
    "descricao_obra",
    "situacao",
    "etapa_obra",
    "percentual_concluido",
    "responsavel_inexecucao",
    "motivo_inexecucao",
    "data_prevista_reinicio",
]

general_info_df['descricao_obra'] = general_info_df['descricao_obra'].apply(clean_text)
general_info_df['situacao'] = general_info_df['situacao'].apply(clean_text)
general_info_df['etapa_obra'] = general_info_df['etapa_obra'].apply(clean_text)
general_info_df['responsavel_inexecucao'] = general_info_df['responsavel_inexecucao'].apply(clean_text)
general_info_df['motivo_inexecucao'] = general_info_df['motivo_inexecucao'].apply(clean_text)

# Converte percentual
general_info_df['percentual_concluido'] = general_info_df["percentual_concluido"].apply(clean_percentage)
# Extrai tipo de modalidade
general_info_df['tipo_modalidade'] = general_info_df['modalidade_licitacao'].apply(extract_modality_type)
# Converte data prevista de reinício
general_info_df['data_prevista_reinicio'] = general_info_df['data_prevista_reinicio'].apply(parse_date)


general_info_df.columns


Index(['modalidade_licitacao', 'descricao_obra', 'situacao', 'etapa_obra',
       'percentual_concluido', 'responsavel_inexecucao', 'motivo_inexecucao',
       'data_prevista_reinicio', 'tipo_modalidade'],
      dtype='object')

In [24]:
# Analisa o rótulo `modalidade_licitação` para definir o tipo da modalidade

# Avaliar todas as variações existentes
modalidade_count = general_info_df['modalidade_licitacao'].value_counts()
modalidade_count.head()

modalidade_licitacao
CP 017/2019    12
CP 016/2022    11
CP 021/2021    10
CP 025/2019     8
CP 002/2019     7
Name: count, dtype: int64

In [25]:
# Identificar padrão e extrair siglas do inicio de texto

In [26]:
# Cria categorias de situação
general_info_df['situacao'] = general_info_df['situacao'].map({
   'Concluída': 'CONCLUIDA',
    'em Andamento': 'EM_ANDAMENTO',
    'Contrato Rescindido': 'RESCINDIDA',
    'Paralisada': 'PARALISADA',
})

In [27]:
# Adicionar na etapa de enriquecimento
# Cria faixas de conclusão
bins = [0, 25, 50, 75, 100]
labals = ['0-25%', '25-50%', '50-75%', '75-100%']

general_info_df['faixa_conclusao'] = pd.cut(
    general_info_df['percentual_concluido'],
    bins=bins,
    labels=labals,
    include_lowest=True
)

general_info_df.head()

,modalidade_licitacao,descricao_obra,situacao,etapa_obra,percentual_concluido,responsavel_inexecucao,motivo_inexecucao,data_prevista_reinicio,tipo_modalidade,faixa_conclusao
0,Concorrência Eletrônica nº 90003/2024,Reforma e requalificação do prédio administrat...,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,12.27,None,None,NaT,CONCORRÊNCIA,0-25%
1,Dispensa 0001/2024,Conclusão da reforma e ampliação da Escola Mun...,EM_ANDAMENTO,"OBRA ESTÁ NA FASE DE ACABAMENTOS, EXECUÇÃO DE ...",84.52,None,None,NaT,DISPENSA,75-100%
2,PREGÃO ELETRÔNICO Nº. 006/2023 - Ata de Regist...,Fornecimento de SISTEMA DE GERAÇÃO DE ENERGIA ...,EM_ANDAMENTO,"ESTÃO SENDO INSTALADOS 800,31 KWp, EQUIVALENTE...",44.00,None,None,NaT,PREGÃO,25-50%
3,DISPENSA ELETRÔNICO Nº. 90045/2025 - PROCESSO ...,Execução dos Serviços Hidráulicos de Combate a...,EM_ANDAMENTO,Execução dos Serviços Hidráulicos de Combate a...,6.00,None,None,NaT,DISPENSA,0-25%
4,CONCORRÊNCIA ELETRÔNICA Nº. 90001/2025 - PROCE...,Pavimentação e sinalização na Rua Professora D...,EM_ANDAMENTO,Pavimentação e sinalização na Rua Professora D...,6.00,None,None,NaT,CONCORRÊNCIA,0-25%


### 2.4 Tratamento dos dados de contrato

In [28]:
contract_df.columns = [
    'numero_contrato_principal',
    'data_inicio',
    'prazo',
    'valor_contratado',
    'data_conclusao_paralisacao',
]


In [29]:
# Limpa textos
contract_df['numero_contrato_principal'] = contract_df['numero_contrato_principal'].apply(clean_text)
contract_df['prazo'] = contract_df['prazo'].apply(clean_text)

In [30]:
# Converte valores monetários
contract_df['valor_contratado'] = contract_df['valor_contratado'].apply(clean_currency)

In [31]:
# Converte data de inicio
contract_df['data_inicio'] = contract_df['data_inicio'].apply(parse_date)

In [32]:
contract_df.head()

,numero_contrato_principal,data_inicio,prazo,valor_contratado,data_conclusao_paralisacao
0,19/2024,2024-07-23,12 MESES,3367764.05,-
1,012/2024,2024-07-09,12 MESES,741995.10,-
2,075/2024,2024-06-27,12 MESES,12000000.00,-
3,172/2025,2025-07-08,2 MESES,116980.00,-
4,089/2025,NaT,"226.403,54",NaN,NaN


### 2.5 Tratamento dos dados de contratados

In [33]:
contracted_df.sample(10)

,numero_contrato_referencia,cnpj_contratada,razao_social_contratada,data_icicio_contratada,data_inicio_contratada
138,090/2022,10.565.011/0001-72,PLANALTO PAJEU EMPREENDIMENTOS - LTDA,None,02/12/2022
101,Contrato Nº 035/2024,04.433.259/0001-87,Talentos Promecc Atacado e Produção de Eventos...,None,31/05/2024
12,026/2024,34.346.587/0001-07,ENGETEC SERVIÇOS DE ENGENHARIA LTDA,None,27/08/2024
281,055/2020,26.383.392/0001-09,P J LOGÍSTICA E CONSTRUÇÃO EIRELI,None,28/08/2020
99,070/2022,26.725.233/0001-45,B & Q CONSTRUTORA E EMPREENDIMENTOS LTDA,None,23/06/2022
159,032/2018,13.838.224/0001-19,CBL EMPREENDIMENTOS LTDA - EPP,None,04/09/2018
105,013/2024,02.520.264/0001-00,CERTEC ESTRUTURAS METÁLICAS LTDA ME,None,25/03/2024
354,03/2019,20.316.425/0001-11,CONSERV EIRELE-ME,None,24/01/2019
330,065/2019,05.244.095/0001-02,CONSTRUTORA BG EIRELE EPP,None,28/08/2019
315,044/2019,13.923.606/0001-40,"PORSAN ENGENHARIA, PROJETOS E CONSULTORIA EIRE...",None,29/05/2019


### 2.6 Tratamento dos dados de despesas do exercícios

In [34]:
fiscal_df.head()

,valor_medio_acumulado,valor_pago_periodo,valor_pago_exercicio
0,"413.074,68","344.629,58","344.629,58"
1,"627.104,97","324.579,24","324.579,24"
2,"5.280.000,00","5.280.000,00","5.280.000,00"
3,0,0,0
4,0,0,0


In [35]:
# Renomeia as colunas
fiscal_df.columns = [
    'valor_medio_acumulado',
    'valor_pago_periodo',
    'valor_pago_exercicio'
]

In [36]:
# Converte valores monetários
fiscal_df['valor_medio_acumulado'] = fiscal_df['valor_medio_acumulado'].apply(clean_currency)
fiscal_df['valor_pago_exercicio'] = fiscal_df['valor_pago_exercicio'].apply(clean_currency)
fiscal_df['valor_pago_periodo'] = fiscal_df['valor_pago_periodo'].apply(clean_currency)

### 2.7 Tratamento dos dados de aditivos

In [37]:
amendment_df.head()

,contract_amendment_prazo aditado,contract_amendment_valor aditado acumulado (r$)
0,-,-
1,-,-
2,-,-
3,-,-
4,-,-


In [38]:
# Renomeia colunas
amendment_df.columns = ['prazo_aditado', 'valor_aditado_acumulado']

In [39]:
# Limpa textos
amendment_df['prazo_aditado'] = amendment_df['prazo_aditado'].apply(clean_text)

In [40]:
amendment_df.head()

,prazo_aditado,valor_aditado_acumulado
0,None,-
1,None,-
2,None,-
3,None,-
4,None,-


In [41]:
amendment_df['valor_aditado_acumulado'] = amendment_df['valor_aditado_acumulado'].apply(clean_currency)

In [42]:
amendment_df.sample(10)

,prazo_aditado,valor_aditado_acumulado
22,None,2349878.52
0,None,NaN
347,None,49124.91
31,60 MESES -14/02/2025,3635150.00
327,None,NaN
122,486 DIAS - 15/06/2025,464950.62
228,None,NaN
224,None,NaN
101,None,NaN
368,31/12/2018 (486 DIAS) - LOTE I 30/01/2019 (396...,1148485.93


### 2.8 Tratamento dos dados de valores pagos acumulados

In [43]:
cumulative_df.head()

,cumulative_amount_paid_valor pago acumulado
0,"344.629,58"
1,"324.579,24"
2,"5.280.000,00"
3,0
4,0


In [44]:
# Renomeira a coluna
cumulative_df.columns = ['valor_pago_acumulado']

In [45]:
cumulative_df['valor_pago_acumulado'] = cumulative_df['valor_pago_acumulado'].apply(clean_currency)

In [46]:
cumulative_df.head()

,valor_pago_acumulado
0,344629.58
1,324579.24
2,5280000.00
3,0.00
4,0.00


### 2.9 Concatenação do DataFrame final

In [47]:
df_final = pd.concat([
    general_info_df,
    contract_df,
    contracted_df,
    fiscal_df,
    amendment_df,
    cumulative_df,
    df[['work_location', 'source_url', 'extraction_date']]
], axis=1
    
)

In [48]:
df_final.shape

(419, 29)

In [49]:
df_final.columns

Index(['modalidade_licitacao', 'descricao_obra', 'situacao', 'etapa_obra',
       'percentual_concluido', 'responsavel_inexecucao', 'motivo_inexecucao',
       'data_prevista_reinicio', 'tipo_modalidade', 'faixa_conclusao',
       'numero_contrato_principal', 'data_inicio', 'prazo', 'valor_contratado',
       'data_conclusao_paralisacao', 'numero_contrato_referencia',
       'cnpj_contratada', 'razao_social_contratada', 'data_icicio_contratada',
       'data_inicio_contratada', 'valor_medio_acumulado', 'valor_pago_periodo',
       'valor_pago_exercicio', 'prazo_aditado', 'valor_aditado_acumulado',
       'valor_pago_acumulado', 'work_location', 'source_url',
       'extraction_date'],
      dtype='object')

### 2.10 Criação da tabela de documentos normalizada

In [50]:
import hashlib

def make_obra_id(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:16] 

df_final["obra_id"] = df_final["source_url"].apply(make_obra_id)

df_final.head()

,modalidade_licitacao,descricao_obra,situacao,etapa_obra,percentual_concluido,responsavel_inexecucao,motivo_inexecucao,data_prevista_reinicio,tipo_modalidade,faixa_conclusao,...,valor_medio_acumulado,valor_pago_periodo,valor_pago_exercicio,prazo_aditado,valor_aditado_acumulado,valor_pago_acumulado,work_location,source_url,extraction_date,obra_id
0,Concorrência Eletrônica nº 90003/2024,Reforma e requalificação do prédio administrat...,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,12.27,None,None,NaT,CONCORRÊNCIA,0-25%,...,413074.68,344629.58,344629.58,None,NaN,344629.58,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/concorrencia-n...,2025-12-22T08:57:17.034673,4a75bc9ecba5793c
1,Dispensa 0001/2024,Conclusão da reforma e ampliação da Escola Mun...,EM_ANDAMENTO,"OBRA ESTÁ NA FASE DE ACABAMENTOS, EXECUÇÃO DE ...",84.52,None,None,NaT,DISPENSA,75-100%,...,627104.97,324579.24,324579.24,None,NaN,324579.24,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/dispensa-001-2...,2025-12-22T08:57:17.045117,89ed9a006f29651d
2,PREGÃO ELETRÔNICO Nº. 006/2023 - Ata de Regist...,Fornecimento de SISTEMA DE GERAÇÃO DE ENERGIA ...,EM_ANDAMENTO,"ESTÃO SENDO INSTALADOS 800,31 KWp, EQUIVALENTE...",44.00,None,None,NaT,PREGÃO,25-50%,...,5280000.00,5280000.00,5280000.00,None,NaN,5280000.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/pregao-eletron...,2025-12-22T08:57:17.053175,d07371eba7e15542
3,DISPENSA ELETRÔNICO Nº. 90045/2025 - PROCESSO ...,Execução dos Serviços Hidráulicos de Combate a...,EM_ANDAMENTO,Execução dos Serviços Hidráulicos de Combate a...,6.00,None,None,NaT,DISPENSA,0-25%,...,0.00,0.00,0.00,None,NaN,0.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/dispensa-eletr...,2025-12-22T08:57:17.060230,bf42b4765e2967c3
4,CONCORRÊNCIA ELETRÔNICA Nº. 90001/2025 - PROCE...,Pavimentação e sinalização na Rua Professora D...,EM_ANDAMENTO,Pavimentação e sinalização na Rua Professora D...,6.00,None,None,NaT,CONCORRÊNCIA,0-25%,...,0.00,0.00,0.00,None,NaN,0.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/concorrencia-e...,2025-12-22T08:57:17.068709,66c46fed16cd433e


In [51]:
all_documents = []

for obra_id, row in df.iterrows():
    obra_url = row["source_url"]
    docs = row["all_documents"]

    # Garante estrutura esperada
    if not isinstance(docs, dict):
        continue

    # Caso novo: se o collector tiver salvado direto a lista em vez do dict completo
    if "paired_data" in docs:
        pares = docs["paired_data"]
    else:
        # fallback: se for diretamente a lista (pouco provável, mas seguro)
        pares = docs

    if not isinstance(pares, list):
        continue

    for pair in pares:
        titulo = pair.get("category")
        url = pair.get("value")

        # pula pares sem URL
        if not url:
            continue

        all_documents.append(
            {
                "obra_id": make_obra_id(row["source_url"]),
                "obra_url": obra_url,
                "documento_titulo": clean_text(titulo),
                "documento_url": str(url).strip(),
            }
        )

documents_df = pd.DataFrame(all_documents)

# Opcional: filtrar só URLs http(s)
#documents_df = documents_df[
#    documents_df["documento_url"].str.startswith("http", na=False)
#]

documents_df.sample(10)

,obra_id,obra_url,documento_titulo,documento_url
1970,5d9b698f82d1e1f4,https://caruaru.pe.gov.br/obras/execucao-dos-s...,Processo na íntegra vol. 2,https://caruaru.pe.gov.br/wp-content/uploads/2...
14,89ed9a006f29651d,https://caruaru.pe.gov.br/obras/dispensa-001-2...,Projeto Básico,https://caruaru.pe.gov.br/wp-content/uploads/2...
1861,a5cfd05296e02611,https://caruaru.pe.gov.br/obras/manutencao-pre...,Contrato,https://caruaru.pe.gov.br/wp-content/uploads/2...
363,904d05413a5b384e,https://caruaru.pe.gov.br/obras/cp-043-2023-co...,Licença Ambiental,https://caruaru.pe.gov.br/wp-content/uploads/2...
2552,7a885cccb62ff0d6,https://caruaru.pe.gov.br/obras/cp-018-2019-co...,Planilha Onerada Allan Galdino,https://caruaru.pe.gov.br/wp-content/uploads/2...
1344,b01d4f1406356307,https://caruaru.pe.gov.br/obras/execucao-de-pa...,Anexos,https://caruaru.pe.gov.br/wp-content/uploads/2...
1869,7b5bb63cd5e9d26b,https://caruaru.pe.gov.br/obras/manutencao-pre...,Edital,https://caruaru.pe.gov.br/wp-content/uploads/2...
1294,9ba69496b20a6674,https://caruaru.pe.gov.br/obras/servicos-de-re...,Composição de Custos Unitários,https://caruaru.pe.gov.br/wp-content/uploads/2...
121,ea96d93d70c39813,https://caruaru.pe.gov.br/obras/cp-044-2022-co...,Composição de Preços Unitários,https://caruaru.pe.gov.br/wp-content/uploads/2...
2488,413613bdc9939233,https://caruaru.pe.gov.br/obras/execucao-do-sa...,Cronograma Físico Financeiro - Travessa Chico ...,https://caruaru.pe.gov.br/wp-content/uploads/2...


##### Criando um hash estável da URL (ids curtos)

In [52]:
df_final.head()

,modalidade_licitacao,descricao_obra,situacao,etapa_obra,percentual_concluido,responsavel_inexecucao,motivo_inexecucao,data_prevista_reinicio,tipo_modalidade,faixa_conclusao,...,valor_medio_acumulado,valor_pago_periodo,valor_pago_exercicio,prazo_aditado,valor_aditado_acumulado,valor_pago_acumulado,work_location,source_url,extraction_date,obra_id
0,Concorrência Eletrônica nº 90003/2024,Reforma e requalificação do prédio administrat...,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,12.27,None,None,NaT,CONCORRÊNCIA,0-25%,...,413074.68,344629.58,344629.58,None,NaN,344629.58,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/concorrencia-n...,2025-12-22T08:57:17.034673,4a75bc9ecba5793c
1,Dispensa 0001/2024,Conclusão da reforma e ampliação da Escola Mun...,EM_ANDAMENTO,"OBRA ESTÁ NA FASE DE ACABAMENTOS, EXECUÇÃO DE ...",84.52,None,None,NaT,DISPENSA,75-100%,...,627104.97,324579.24,324579.24,None,NaN,324579.24,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/dispensa-001-2...,2025-12-22T08:57:17.045117,89ed9a006f29651d
2,PREGÃO ELETRÔNICO Nº. 006/2023 - Ata de Regist...,Fornecimento de SISTEMA DE GERAÇÃO DE ENERGIA ...,EM_ANDAMENTO,"ESTÃO SENDO INSTALADOS 800,31 KWp, EQUIVALENTE...",44.00,None,None,NaT,PREGÃO,25-50%,...,5280000.00,5280000.00,5280000.00,None,NaN,5280000.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/pregao-eletron...,2025-12-22T08:57:17.053175,d07371eba7e15542
3,DISPENSA ELETRÔNICO Nº. 90045/2025 - PROCESSO ...,Execução dos Serviços Hidráulicos de Combate a...,EM_ANDAMENTO,Execução dos Serviços Hidráulicos de Combate a...,6.00,None,None,NaT,DISPENSA,0-25%,...,0.00,0.00,0.00,None,NaN,0.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/dispensa-eletr...,2025-12-22T08:57:17.060230,bf42b4765e2967c3
4,CONCORRÊNCIA ELETRÔNICA Nº. 90001/2025 - PROCE...,Pavimentação e sinalização na Rua Professora D...,EM_ANDAMENTO,Pavimentação e sinalização na Rua Professora D...,6.00,None,None,NaT,CONCORRÊNCIA,0-25%,...,0.00,0.00,0.00,None,NaN,0.00,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/concorrencia-e...,2025-12-22T08:57:17.068709,66c46fed16cd433e


In [53]:
df_final[df_final["obra_id"] == "e1889205147d606b"]

,modalidade_licitacao,descricao_obra,situacao,etapa_obra,percentual_concluido,responsavel_inexecucao,motivo_inexecucao,data_prevista_reinicio,tipo_modalidade,faixa_conclusao,...,valor_medio_acumulado,valor_pago_periodo,valor_pago_exercicio,prazo_aditado,valor_aditado_acumulado,valor_pago_acumulado,work_location,source_url,extraction_date,obra_id
241,CP 015/2021,"Reforma e ampliação de unidades de ensino, loc...",CONCLUIDA,Finalizada,90.75,None,None,NaT,CARTA PROPOSTA,75-100%,...,194488.14,NaN,NaN,02/10/2022 (30 Dias) - Concluída,42240.12,194488.14,[https://caruaru.pe.gov.br/wp-content/uploads/...,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,2025-12-22T08:57:20.081020,e1889205147d606b


In [54]:
documents_df[documents_df["obra_id"] == "e1889205147d606b"]

,obra_id,obra_url,documento_titulo,documento_url
1535,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Edital,https://caruaru.pe.gov.br/wp-content/uploads/2...
1536,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Contrato,https://caruaru.pe.gov.br/wp-content/uploads/2...
1537,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Projeto Básico,https://caruaru.pe.gov.br/wp-content/uploads/2...
1538,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Anexos,https://caruaru.pe.gov.br/wp-content/uploads/2...


##### Exemplo de cruzamento

In [55]:
cols = ["obra_id", "source_url", "cnpj_contratada", "numero_contrato_principal", "situacao", "etapa_obra", "valor_contratado", "data_conclusao_paralisacao",]
documents_df = documents_df.merge(
    df_final[cols],
    on="obra_id",
    how="left")

documents_df.head()

,obra_id,obra_url,documento_titulo,documento_url,source_url,cnpj_contratada,numero_contrato_principal,situacao,etapa_obra,valor_contratado,data_conclusao_paralisacao
0,4a75bc9ecba5793c,https://caruaru.pe.gov.br/obras/concorrencia-n...,Projeto Básico,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/concorrencia-n...,00.654.704/0001-88,19/2024,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,3367764.05,-
1,4a75bc9ecba5793c,https://caruaru.pe.gov.br/obras/concorrencia-n...,Edital,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/concorrencia-n...,00.654.704/0001-88,19/2024,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,3367764.05,-
2,4a75bc9ecba5793c,https://caruaru.pe.gov.br/obras/concorrencia-n...,Contrato,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/concorrencia-n...,00.654.704/0001-88,19/2024,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,3367764.05,-
3,4a75bc9ecba5793c,https://caruaru.pe.gov.br/obras/concorrencia-n...,Memória de Cálculo,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/concorrencia-n...,00.654.704/0001-88,19/2024,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,3367764.05,-
4,4a75bc9ecba5793c,https://caruaru.pe.gov.br/obras/concorrencia-n...,Licença Ambiental,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/concorrencia-n...,00.654.704/0001-88,19/2024,EM_ANDAMENTO,OBRA ESTÁ NA FASE DE EXECUÇÃO DE ALVENARIAS E ...,3367764.05,-


In [56]:
documents_df[documents_df["obra_id"] == "e1889205147d606b"]

,obra_id,obra_url,documento_titulo,documento_url,source_url,cnpj_contratada,numero_contrato_principal,situacao,etapa_obra,valor_contratado,data_conclusao_paralisacao
1535,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Edital,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,30.651.938/0001-32,025/2021,CONCLUIDA,Finalizada,172077.08,02/09/2022
1536,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Contrato,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,30.651.938/0001-32,025/2021,CONCLUIDA,Finalizada,172077.08,02/09/2022
1537,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Projeto Básico,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,30.651.938/0001-32,025/2021,CONCLUIDA,Finalizada,172077.08,02/09/2022
1538,e1889205147d606b,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,Anexos,https://caruaru.pe.gov.br/wp-content/uploads/2...,https://caruaru.pe.gov.br/obras/reforma-e-ampl...,30.651.938/0001-32,025/2021,CONCLUIDA,Finalizada,172077.08,02/09/2022


In [57]:
# Exporta df_final e documents_df_final para csv
df_final.to_csv('obras_publicas_dataset.csv', index=False)
documents_df.to_csv("documents_dataset.csv", index=False)

## 3. Análise de qualidade de dados

### 3.1. Análise de completude

In [58]:
from collections import Counter

In [59]:
def analisar_completude(df):
    """
    Identifica colunas com dados faltantes, nulos ou vazios.
    Retorna uma DataFrame com métricas de qualidade por coluna
    """

    resultado = []

    for coluna in df.columns:
        total = len(df)
        nulos = df[coluna].isna().sum()

        # Para colunas de textos, verifaca vazios
        vazios = 0
        if df[coluna].dtype == 'object':
            vazios = df[coluna].astype(str).apply(
                lambda x: x.strip() in ['', '-', 'NA', 'None']
            ).sum()

    validos = total - nulos - vazios
    pct_completud = (validos / total) * 100

    resultado.append({
        'coluna': coluna,
        'validos': validos,
        'nulos': nulos,
        'vazios': vazios,
        'pct_completude': round(pct_completud, 2)
    })

    return pd.DataFrame(resultado)



### 3.2. Detecção de campos misturados 

In [60]:
def detectar_campos_misturados(df, coluna):
    """
    Identifica se uma coluna contém múltiplos tipos de dados minsturados.
    Útil para campos que misturam data + texto narrativo.
    """

    serie = df[coluna].dropna().astype(str)

    # Contar padrões
    tem_datas = serie.str.contains(r'\d{2}/\d{2}/\d{4}', regex=True).sum()
    tem_parenteses = serie.str.contains(r'\([^)]+\)', regex=True).sum()
    tem_uppercase = serie.str.contains(r'[A-Z]{3,}', regex=True).sum()

    # Palavras-chave comuns em obras públicas
    palavras = ['RESCISÃO', 'RESCISAO', 'PARALISAÇÃO', 'PARALISACAO', 'REINICÍCIO', 'REINICIO', 'TERMO', 'UNILATERAL'] 

    palavras_encontradas = {}

    for palavra in palavras:
        count = serie.str.contains(palavra, case=False, na=False).sum()
        if count > 0:
            palavras_encontradas[palavra] = count

    # Calcula o score de mistura
    score = 0
    if tem_datas > 0: score += 20
    if tem_parenteses > 0: score +=30
    if len(palavras_encontradas) > 0: score += 25
    if len(serie.unique()) / len(serie) > 0.5: score += 25

    return {
        'coluna': coluna,
        'score_mistura': score,
        'tem_datas': tem_datas,
        'tem_parenteses': tem_parenteses,
        'palavras_chave': palavras_encontradas,
        'total_valores': len(serie),
        'valores_unicos': len(serie.unique()),
    }

### 3.3. Categorização de valores misturados

In [61]:
def categorizar_campo_data(serie):
    '''
    Categoriza cada valor de um campo de data identificando seu padrão.
    Retorna série com categorias atribuídas.
    '''
    def categorizar(valor):
        if pd.isna(valor) or str(valor).strip() in ['', '-']:
            return 'VAZIO_OU_HIFEN'

        valor_str = str(valor).upper()

        if 'RESCISÃO' in valor_str or 'RESCISAO' in valor_str:
            return 'DATA_COM_TERMO_RESCISAO'
        elif 'PARALISAÇÃO' in valor_str or 'PARALISACAO' in valor_str:
            if 'REINÍCIO' in valor_str or 'REINICIO' in valor_str:
                return 'HISTORICO_COMPLETO'
            else:
                return 'DATA_COM_PARALISACAO'
        elif re.match(r'^\d{2}/\d{2}/\d{4}$', str(valor).strip()):
            return 'DATA_SIMPLES'
        else:
            return 'OUTRO_FORMATO'

    return serie.apply(categorizar)

### 3.4. Análise de inconsistências

In [62]:
def analisar_inconsistencias_status(df):
    '''
    Identifica obras onde status declarado não condiz com indicadores
    de execução física e financeira.
    '''
    inconsistencias = []

    for idx, row in df.iterrows():
        flags = []
        descricoes = []

        situacao = str(row.get('situacao', '')).upper()
        percentual = row.get('percentual_concluido', 0)
        valor_contratado = row.get('valor_contratado', 0)
        valor_pago = row.get('valor_pago_acumulado', 0)

        # Converte percentual se necessário
        if isinstance(percentual, str):
            try:
                percentual = float(percentual.replace('%', '').replace(',', '.'))
            except:
                percentual = 0

        # Calcula execução financeira
        perc_exec_financeira = 0
        if pd.notna(valor_contratado) and pd.notna(valor_pago) and valor_contratado > 0:
            perc_exec_financeira = (valor_pago / valor_contratado) * 100

        # Checa inconsistências
        if 'RESCIND' in situacao:
            if pd.notna(percentual) and percentual >= 95:
                flags.append('FISICA_COMPLETA')
                descricoes.append(f"Rescindida mas {percentual:.1f}% concluída")

            if perc_exec_financeira >= 95:
                flags.append('FINANCEIRA_COMPLETA')
                descricoes.append(f"Rescindida mas {perc_exec_financeira:.1f}% executada")

        if perc_exec_financeira > 150:
            flags.append('VALOR_EXCEDIDO')
            descricoes.append(f"Valor pago >> valor contratado")

        if flags:
            inconsistencias.append({
                'index': idx,
                'numero_contrato': row.get('numero_contrato'),
                'situacao': row.get('situacao'),
                'percentual_concluido': percentual,
                'perc_exec_financeira': perc_exec_financeira,
                'flags': flags,
                'descricoes': descricoes,
                'gravidade': len(flags)
            })

    return pd.DataFrame(inconsistencias)

In [63]:
print("\n1. ANÁLISE DE COMPLETUDE")

df_completude = analisar_completude(df_final)
colunas_problematicas = df_completude[df_completude['pct_completude'] < 80]
print(f"Colunas problemáticas: {len(colunas_problematicas)}")
print(colunas_problematicas[['coluna', 'pct_completude']])

# 3. Detecta campos misturados
print("\n2. CAMPOS COM DADOS MISTURADOS")
if 'data_conclusao_paralisacao' in df_final.columns:
    resultado_mistura = detectar_campos_misturados(df_final, 'data_conclusao_paralisacao')
    # resultado_mistura = detectar_campos_misturados(df_final, 'data_prevista_reinicio')
    print(f"Score de mistura: {resultado_mistura['score_mistura']}/100")
    print(f"Registros com datas: {resultado_mistura['tem_datas']}")
    print(f"Palavras-chave encontradas: {resultado_mistura['palavras_chave']}")

    # Categoriza valores
    categorias = categorizar_campo_data(df_final['data_conclusao_paralisacao'])
    # categorias = categorizar_campo_data(df_final['data_prevista_reinicio'])
    print("\nDistribuição de formatos:")
    print(categorias.value_counts())

# 4. Identifica inconsistências
print("\n3. INCONSISTÊNCIAS STATUS vs INDICADORES")
df_inconsist = analisar_inconsistencias_status(df_final)
if len(df_inconsist) > 0:
    print(f"Inconsistências detectadas: {len(df_inconsist)} ({len(df_inconsist)/len(df)*100:.1f}%)")
    print("\nCasos mais graves:")
    print(df_inconsist.nlargest(3, 'gravidade')[['numero_contrato', 'gravidade', 'descricoes']])
else:
    print("✓ Nenhuma inconsistência detectada")

# 5. Relatório final
print("\n4. RESUMO EXECUTIVO")
print(f"- Campos misturados: {resultado_mistura.get('tem_datas', 0)} casos")
print(f"- Históricos complexos: {(categorias == 'HISTORICO_COMPLETO').sum()} casos")
print(f"- Inconsistências: {len(df_inconsist)} casos")
print(f"- Dados faltantes: {(categorias == 'VAZIO_OU_HIFEN').sum()} casos")


1. ANÁLISE DE COMPLETUDE
Colunas problemáticas: 0
Empty DataFrame
Columns: [coluna, pct_completude]
Index: []

2. CAMPOS COM DADOS MISTURADOS
Score de mistura: 100/100
Registros com datas: 307
Palavras-chave encontradas: {'RESCISÃO': np.int64(14), 'PARALISAÇÃO': np.int64(96), 'REINICIO': np.int64(1), 'TERMO': np.int64(6), 'UNILATERAL': np.int64(3)}

Distribuição de formatos:
data_conclusao_paralisacao
OUTRO_FORMATO              130
VAZIO_OU_HIFEN             105
HISTORICO_COMPLETO          78
DATA_SIMPLES                77
DATA_COM_PARALISACAO        15
DATA_COM_TERMO_RESCISAO     14
Name: count, dtype: int64

3. INCONSISTÊNCIAS STATUS vs INDICADORES
Inconsistências detectadas: 47 (11.2%)

Casos mais graves:
   numero_contrato  gravidade  \
40            None          2   
0             None          1   
1             None          1   

                                           descricoes  
40  [Rescindida mas 95.9% concluída, Rescindida ma...  
0                    [Valor pago >> 